# Bhoonidhi SDK — auth

Using Bhoonidhi from Python. One import, one `BhoonidhiClient` object;
every method matches a `bhd` CLI command.

This talks to the live portal. Run the cells top to bottom and enter your
real credentials when prompted.

## 1. Import and create a client

In [1]:
from bhoonidhi_downloader.sdk import BhoonidhiClient, BhoonidhiError

client = BhoonidhiClient()
client

## 2. Log in

`client.login(...)` matches `bhd auth login`. The password is read with `getpass`, so it isn't saved in the notebook.

Some accounts also require a 6-digit email OTP after the password. Pass `otp_prompt` to ask for it interactively, as below -- it's only called if the account actually needs one.

In [ ]:
import getpass

def get_otp(message: str) -> str:
    print(message)
    return input("Enter the 6-digit OTP: ")

username = input("Bhoonidhi username: ")
password = getpass.getpass("Bhoonidhi password: ")

# otp_prompt is only called if the account needs it -- a password-only
# account still logs in on the first request, with no prompt shown.
client.login(username, password, otp_prompt=get_otp)
print("authenticated:", client.is_authenticated)

### Non-interactive login (scripts, cron, CI)

If you already have the OTP in hand -- or the account is password-only and never needs one -- pass it directly instead of prompting:

In [ ]:
# client.login(username, password, otp="123456")

## 3. whoami

Matches `bhd auth whoami`.

In [3]:
client.whoami()

'geovicco'

## 4. status

Matches `bhd auth status`. Returns the session and whether its token still validates against the portal.

In [4]:
session, is_valid = client.status()
print("token valid:", is_valid)
print("username:   ", session.username)

token valid: True
username:    geovicco


## 5. refresh

Matches `bhd auth refresh`. Renews the token without re-entering the password (only works while the token is still fresh).

In [5]:
client.refresh()
print("token still valid:", client.status()[1])

token still valid: True


## 6. Error handling

A bad login raises `BhoonidhiError`.

In [6]:
try:
    BhoonidhiClient().login("not-a-real-user", "wrong-password")
except BhoonidhiError as e:
    print(f"{type(e).__name__}: {e}")

BhoonidhiAuthError: Login failed. Reason: THE USER ID AND PASSWORD ARE NOT CORRECT. Both the fields are Case Sensitive


## 7. logout

Matches `bhd auth logout`. Clears the saved session.

In [7]:
print("logged out:", client.logout())
print("authenticated:", client.is_authenticated)

logged out: True
authenticated: False
